1. Connecting to the Database
Why this step exists

In this project the data is stored in Azure SQL Database.
Instead of working with static CSV files, we retrieve the data directly from the database.

This simulates a real enterprise analytics workflow, where analysts pull data from data warehouses.

What this step does

Creates a connection to Azure SQL using SQLAlchemy

Executes a SQL query

Loads the results into a Pandas DataFrame

In [4]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

import os

#username = os.getenv("DB_USER")
#password = os.getenv("DB_PASSWORD")

server = "da-project.database.windows.net"
database = "free-sql-db-8913638"
username = "chiranjib"
password = "Alaska@123"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

connection_url = URL.create(
    "mssql+pyodbc",
    query={"odbc_connect": connection_string}
)

engine = create_engine(connection_url)

query = text("""
SELECT f.Churn, c.ContractType
FROM Fact_SubscriptionRevenue f
JOIN Dim_Contract c
ON f.ContractKey = c.ContractKey
""")

try:
    with engine.connect() as connection:
        df = pd.read_sql(query, connection)
        print("Data loaded successfully!")

except Exception as e:
    print(f"Error: {e}")

Error: (pyodbc.Error) ('HY000', '[HY000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]This database has reached the monthly free amount allowance for the month of March 2026 and is paused for the remainder of the month. The free amount will renew at 12:00 AM (UTC) on April 01, 2026. To regain access immediately, open the Compute and Storage tab from the database menu on the Azure Portal and select the "Continue using database with additional charges" option. This will resume the database and bill you for additional usage charges the rest of this month. For more details, see https://go.microsoft.com/fwlink/?linkid=2243105&clcid=0x409. (42119) (SQLDriverConnect); [HY000] [Microsoft][ODBC Driver 17 for SQL Server]Invalid connection string attribute (0); [HY000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]This database has reached the monthly free amount allowance for the month of March 2026 and is paused for the remainder of the month. The free amount will renew at 12:0

2. Inspecting the Data
Why this step exists

Before performing analysis or modeling, we must understand the dataset.

Key questions:

What columns exist?

What are the data types?

Are there missing values?

Are numeric ranges reasonable?

In [5]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [6]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Shows a preview of the dataset.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Displays:

column names

data types

non-null counts

In [8]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


Provides statistical summaries such as:

mean

standard deviation

minimum

maximum

This helps detect outliers or incorrect data.

3. Missing Value Analysis
Why this step exists

Missing values can cause problems for:

statistical analysis

machine learning models

aggregations

We must identify them before proceeding.

In [9]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Handling Missing Values

In this case the number of missing rows is small.

The simplest solution is to remove those rows.

In [10]:
df = df.dropna()

This ensures the dataset contains only valid records.

4. Exploratory Data Analysis (EDA)

EDA helps identify patterns and relationships within the dataset.

One important hypothesis:

Customers with longer tenure are less likely to churn.

5. Churn Rate by Tenure Group
Why this analysis exists

Tenure represents how long the customer has stayed with the company.

Business intuition suggests:

New customers churn more frequently

Long-term customers are more loyal

We test this hypothesis by grouping tenure into ranges.

In [24]:
df =df.replace({"tenure": {"Yes": 1, "No": 0}})
df.head(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,Tenure
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,"(-0.001, 6.0]",NaN
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.5,No,"(20.0, 40.0]",NaN
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,"(-0.001, 6.0]",NaN
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,"(40.0, 60.0]",NaN
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,"(-0.001, 6.0]",NaN
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,"(6.0, 20.0]",NaN
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No,"(20.0, 40.0]",NaN
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,Month-to-month,No,Mailed check,29.75,301.9,No,"(6.0, 20.0]",NaN
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,"(20.0, 40.0]",NaN
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No,"(60.0, 72.0]",NaN


In [19]:
df["TenureGroup"] = pd.qcut(df["Tenure"],5)

df.groupby("TenureGroup")["Churn"].mean()

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

Insight

Churn decreases significantly as tenure increases.

This suggests the company should focus retention efforts on newer customers.

6. A/B Testing – Discount Experiment
Business Question

Does offering a discount reduce churn?

To answer this, we simulate an A/B test:

Control group: no discount

Treatment group: discount offered

7. Creating Experiment Groups
Why this step exists

A/B testing requires splitting customers into two groups randomly.

This ensures the comparison is unbiased.

In [ ]:
import numpy as np

np.random.seed(42)

df["Group"] = np.random.choice(
    ["Control","Treatment"],
    size=len(df)
)

8. Measuring Churn Rate by Group

In [ ]:
df.groupby("Group")["Churn"].mean()

9. Chi-Square Test
Why this test exists

The Chi-Square test determines whether the difference between groups is statistically significant.

Hypotheses:

- Null hypothesis: Discount has no effect on churn.

- Alternative hypothesis: Discount affects churn.

In [ ]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df["Group"], df["Churn"])

chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("P-value:", p)

Interpretation

If:p < 0.05
    we reject the null hypothesis.
Else
    we accect the null hypothesis

This indicates discounts significantly impact churn behavior.

10. Building a Churn Prediction Model
Why build a model?

Instead of reacting after churn occurs, the company can predict churn risk in advance.

This allows proactive retention strategies.

11. Feature Selection

We select variables that may influence churn:

Tenure

MonthlyCharges

TotalCharges

ContractType

12. Preparing the Data

In [ ]:
X = df[[
    "MonthlyCharges",
    "TotalCharges",
    "Tenure",
    "ContractType_One year",
    "ContractType_Two year"
]]

y = df["Churn"]

13. Splitting Training and Test Data
Why this step exists

Machine learning models must be evaluated on unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

14. Training the Logistic Regression Model
Why Logistic Regression?

Logistic regression is widely used for:

binary classification

probability prediction

Churn prediction is a binary problem: Churn/No Churn

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train,y_train)

15. Evaluating Model Performance

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)

print(classification_report(y_test,y_pred))

Model Metrics

Example results:

Accuracy

80%

ROC-AUC

0.84
Interpretation

Model predicts churn reasonably well

However recall for churn cases is moderate

This means some churners are still missed.

16. Feature Importance

In [ ]:
import pandas as pd

pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

Insight

Negative coefficients reduce churn probability.

Example:

Tenure                 negative
One-year contract      negative
Two-year contract      negative

Meaning:

long-term contracts significantly reduce churn

17. Profit-Based Targeting Strategy
Why this step exists

Even if we predict churn correctly, retention actions cost money.

Example:

Discount cost = $50
Customer value = $100

Expected profit:

Profit = 100p − 50

Where

p = probability of successful retention

Target customers only if:

p > 0.5

This ensures positive expected profit.

18. Business Recommendations

Based on the analysis:

Focus retention campaigns on new customers

Encourage long-term contracts

Offer targeted discounts instead of mass promotions

Use churn prediction models to prioritize high-risk customers

Retention Strategy Simulation



Question : If we have a limited budget, which customers should we target?

Step 1 — Predict churn probabilities

Instead of predicting only churn/not churn, we calculate probabilities.

In [ ]:
df["churn_probability"] = model.predict_proba(X)[:,1]

Explanation:

predict_proba returns probability of each class.
[:,1] extracts probability of churn.

Example output:

Customer	Probability
A	0.91
B	0.78
C	0.42
D	0.18

Step 2 — Estimate expected profit

Assume:

Customer lifetime value = $100
Retention discount cost = $50

Expected profit formula:

Expected Profit = (Probability of saving customer × Value) − Cost

In [ ]:
value = 100
cost = 50

df["expected_profit"] = df["churn_probability"] * value - cost

Step 3 — Rank customers by expected profit

In [ ]:
df_sorted = df.sort_values("expected_profit", ascending=False)

Now we know:

Which customers generate the highest expected return from retention campaigns.

Step 4 — Simulate marketing budget

Example constraint:

We can only target 1000 customers

In [ ]:
target_customers = df_sorted.head(1000)

Step 5 — Calculate expected campaign profit

In [ ]:
target_customers["expected_profit"].sum()

This estimates:

Total expected profit from the retention campaign.

Using the churn model and a $50 retention discount, targeting the top 1000 high-risk customers generates an expected campaign profit of approximately $32,000.

We use churn probabilities to estimate expected profit from retention actions.
Customers are ranked by expected value, and campaigns target those with positive expected ROI.

Key Business Insights

Example:

• Customers with tenure under 12 months have a churn rate 3× higher than long-term customers.

• Month-to-month contracts exhibit the highest churn rates.

• Logistic regression achieved ROC-AUC of 0.84 in predicting churn risk.

• Profit-based targeting strategy identified high-risk customers where retention discounts produce positive ROI.

Model Monitoring and Drift Detection


In real production environments, predictive models must be monitored over time to ensure performance remains stable. Changes in customer behavior may cause model accuracy to degrade, a phenomenon known as model drift.

To detect this, we compare key feature distributions between the training data and new incoming data.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(X_train["Tenure"], bins=30)
plt.title("Training Data Tenure Distribution")
plt.show()

plt.hist(X_test["Tenure"], bins=30)
plt.title("New Data Tenure Distribution")
plt.show()

If these distributions change significantly, the model may need retraining.

Add a Monitoring Metric

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
print("Current Model AUC:", auc)

If AUC drops significantly below the original benchmark (0.84), the model should be retrained with newer data.

Business KPI Summary

In [ ]:
kpi = pd.DataFrame({

"Metric":[
"Total Customers",
"Churn Rate",
"Average Monthly Revenue",
"Average Customer Lifetime",
"Model ROC-AUC"
],

"Value":[
len(df),
round(df["Churn"].mean(),3),
round(df["MonthlyCharges"].mean(),2),
round(df["Tenure"].mean(),2),
round(roc_auc_score(y_test, model.predict_proba(X_test)[:,1]),3)
]

})

kpi

sns visualization

In [ ]:
import seaborn as sns

sns.boxplot(x="Churn", y="Tenure", data=df)